# 4장 · 크로스 엔트로피 — 틀린 지도의 비용 🗺️

웹 챕터 **"크로스 엔트로피"** 와 짝을 이루는 Python 실습입니다.
웹 실험실에서 잡은 감(q = p 에서 최소)을, 여기서는 **무작위 지도 1만 장 검증과
LLM 손실 미니 시뮬레이션**으로 굳힙니다.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch4')

## 1. 크로스 엔트로피 함수: 그리고 지도 세 장의 요금표

$H(p,q) = \sum p \cdot \log_2 \frac{1}{q}$. 빈도는 세상 p 가, 코드 길이는 내 지도 q 가 정합니다.
일어날 수 있는 일(p>0)에 q=0 을 주면 비용은 무한대.

In [ ]:
import numpy as np

def entropy(p):
    """H(p) — 3장의 그 함수."""
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(np.sum(p * np.log2(1 / p)))

def cross_entropy(p, q):
    """H(p,q) — 세상 p 를 지도 q 로 부호화한 평균 비용 (비트)."""
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    mask = p > 0
    if np.any(q[mask] <= 0):
        return float('inf')   # 일어날 수 있는 일에 확률 0을 준 지도
    return float(np.sum(p[mask] * np.log2(1 / q[mask])))

p_world = np.array([1/2, 1/4, 1/8, 1/8])          # 맑음·흐림·비·눈
maps = {
    '🎯 세상과 같은 지도 (q=p)': p_world,
    '⚖️ 균등 지도             ': np.array([1/4, 1/4, 1/4, 1/4]),
    '🙃 거꾸로 지도           ': np.array([1/8, 1/8, 1/4, 1/2]),
    '💥 눈이 없는 지도        ': np.array([0.5, 0.3, 0.2, 0.0]),
}

H_p = entropy(p_world)
print(f'바닥 H(p) = {H_p:.3f} 비트\n')
for name, q in maps.items():
    c = cross_entropy(p_world, q)
    extra = '∞' if np.isinf(c) else f'+{c - H_p:.3f}'
    print(f'{name} H(p,q) = {c:8.3f} 비트  (초과분 {extra})')
    assert c >= H_p - 1e-9, '바닥보다 싼 지도가 나왔다?! (그럴 수 없음)'

## 2. 무작위 지도 10,000장: 바닥에 도전

지도를 무작위로 10,000장 만들어 요금을 재 봅니다. 단 한 장이라도 H(p) = 1.75비트보다
싸면 교과서를 다시 써야 합니다. 그리고 **요금이 가장 쌌던 지도**는 어떻게 생겼을까요?

In [ ]:
import matplotlib.pyplot as plt

# 한글 폰트 설정 (안 하면 한글 제목/라벨이 네모로 깨짐)
# Windows: 'Malgun Gothic' / macOS: 'AppleGothic' / Linux: 'NanumGothic' 로 바꾸세요
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False  # 음수 부호(-) 깨짐 방지

rng = np.random.default_rng(42)
raw = rng.random((10000, 4)) + 1e-12
qs = raw / raw.sum(axis=1, keepdims=True)
costs = np.array([cross_entropy(p_world, q) for q in qs])

best = qs[costs.argmin()]
print(f'가장 싼 지도의 요금 : {costs.min():.4f} 비트  (바닥 H(p) = {H_p:.4f})')
print(f'가장 싼 지도의 모양 : {np.round(best, 3)}')
print(f'세상 p 의 모양      : {p_world}')
print('→ 요금을 낮출수록 지도가 세상을 닮아갑니다!')
assert costs.min() >= H_p - 1e-9

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(costs, bins=60, color='#6366f1', alpha=0.85)
ax.axvline(H_p, color='#ef4444', lw=2, linestyle='--', label=f'바닥 H(p) = {H_p}비트 (q=p)')
ax.set_xlabel('요금 H(p,q) (비트)')
ax.set_ylabel('지도 개수')
ax.set_title('무작위 지도 10,000장의 요금: 아무도 빨간 바닥선보다 쌀 수 없다')
ax.legend()
plt.show()

## 3. 원핫이면 −log q(정답): 분류기의 채점표

정답이 하나로 정해진 원핫 세상에서는 공식이 정답 항 하나로 줄어듭니다.
고양이/개/새 3지선다 분류기를 채점해 봅시다.

In [ ]:
classes = ['고양이', '개', '새']
p_onehot = np.array([1.0, 0.0, 0.0])   # 정답: 고양이

predictions = {
    '확신하고 맞힘 (0.95, 0.03, 0.02)': np.array([0.95, 0.03, 0.02]),
    '애매하게 맞힘 (0.50, 0.30, 0.20)': np.array([0.50, 0.30, 0.20]),
    '갈팡질팡      (0.34, 0.33, 0.33)': np.array([0.34, 0.33, 0.33]),
    '확신하고 틀림 (0.01, 0.98, 0.01)': np.array([0.01, 0.98, 0.01]),
}

print('예측 (고양이, 개, 새)            | H(p,q) = −log₂ q(정답)')
print('-' * 60)
for name, q in predictions.items():
    full = cross_entropy(p_onehot, q)      # 공식 전체로 계산
    shortcut = -np.log2(q[0])              # 지름길: 정답 항만
    assert abs(full - shortcut) < 1e-9     # 둘이 같은지 확인
    print(f'{name} | {full:6.3f} 비트')

print('\n확신하고 틀리면(정답에 q=0.01) 6.6비트 폭발. 자신 있는 오답이 가장 크게 혼납니다.')

## 4. 크로스 엔트로피 vs MSE: 1부의 복선 풀기

1부 캡스톤은 MSE 를 썼죠. 사실 log 를 피하려던 우회였습니다.
정답에 준 확률 q 에 따라 두 손실이 얼마나 다르게 벌주는지 비교합니다.

In [ ]:
q = np.linspace(0.005, 1, 400)
ce = np.log2(1 / q)        # 크로스 엔트로피 (비트)
mse = (1 - q) ** 2         # MSE (원핫 정답 기준)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(q, ce, color='#6366f1', lw=2.5, label='크로스 엔트로피 -log2 q')
ax.plot(q, mse, color='#f59e0b', lw=2.5, label='MSE (1 - q)^2')
ax.axvline(0.01, color='#94a3b8', lw=1, linestyle=':')
ax.annotate('q=0.01 (확신에 찬 오답)\nCE 약 6.6비트 폭발\nMSE 약 0.98 미지근', (0.01, 6.6),
            textcoords='offset points', xytext=(14, -10), fontsize=10)
ax.set_xlabel('모델이 정답에 준 확률 q')
ax.set_ylabel('손실')
ax.set_ylim(0, 8)
ax.set_title('자신 있는 오답을 벌하는 강도: CE 는 폭발, MSE 는 미지근')
ax.grid(True, color='#eef2f7')
ax.legend()
plt.show()

## 5. LLM 미니 시뮬레이션: 문장 하나의 학습 손실

LLM 학습은 **토큰 위치마다** "다음 토큰 맞히기"를 채점해 평균냅니다.
가상의 문장에서 모델이 각 정답 토큰에 줬던 확률로 손실을 계산해 봅시다.

In [ ]:
tokens =   ['나는', '아침', '에',   '따뜻한', '국밥',  '을',   '먹었다']
q_correct = [0.20,  0.30,  0.95,  0.10,    0.02,   0.90,  0.60]   # 모델이 각 정답 토큰에 준 확률

losses = -np.log2(q_correct)
avg = losses.mean()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(tokens, losses, color=['#f97316' if l > 3 else '#6366f1' for l in losses], alpha=0.9)
ax.bar_label(bars, fmt='%.2f')
ax.axhline(avg, color='#10b981', lw=2, linestyle='--', label=f'평균 = 학습 손실 {avg:.2f}비트')
ax.set_ylabel('토큰별 손실 -log2 q(정답) (비트)')
ax.set_title('문장 하나의 크로스 엔트로피: 예측이 어려운 토큰(국밥)이 손실을 지배한다')
ax.legend()
plt.show()

print(f'토큰별 손실 합 {losses.sum():.2f}비트, 평균 {avg:.2f}비트')
print('훈련 로그의 loss 곡선이 바로 이 평균(단위만 nat)이고, 2^평균 이 7장의 퍼플렉시티입니다.')

## 6. 직접 풀어보기 ✏️

거꾸로 지도의 요금: 세상 p = (1/2, 1/4, 1/8, 1/8), 지도 q = (1/8, 1/8, 1/4, 1/2) 일 때
H(p,q) 는 몇 비트일까요? (웹 챕터의 어려움 문제와 같은 계산)

In [ ]:
# TODO: 아래 answer 를 여러분의 계산으로 바꾸고 실행하세요.
# 힌트: 항별로 p × log₂(1/q) — 또는 cross_entropy(p_world, ...) 재사용

answer = cross_entropy([1/2, 1/4, 1/8, 1/8], [1/8, 1/8, 1/4, 1/2])
print('내 답:', answer)
assert abs(answer - 2.625) < 1e-9, '다시 계산해 보세요 — 1.5 + 0.75 + 0.25 + 0.125'
print('✅ 잘했어요! 바닥(1.75)보다 0.875비트 비싼 최악의 지도. 이 초과분이 5장의 KL 입니다.')

---
**오늘의 한 줄**: H(p,q) = 틀린 지도 요금 전체. q = p 에서 최소(= H(p)),
원핫이면 −log q(정답). 이것이 분류기와 LLM 의 손실함수.

다음 챕터에서는 요금 중 **바가지 초과분만** 떼어내 이름을 붙입니다.
RLHF 페널티와 증류의 언어, **KL 발산 D(p‖q)**. 🙂